# Kalshi vs. Polymarket: Regulated vs. Unregulated Signal Quality

Grounds `paper/sections/5_polymarket_comparison.md`. Reproduces the FEDS paper's Figure-1-style *forecast-error-by-days-before-FOMC* chart, but with a **Polymarket** series added alongside Kalshi on the same FOMC rate-decision outcome, using the *same* pipeline (`kalshi_utils.ladder_to_pdf` -> mean/median/mode).

The FEDS paper mentions Polymarket only once, to dismiss it as operating "in a legal gray area" with lower liquidity and looser limits (p.8); it never runs a side-by-side. This notebook builds the scaffolding for that comparison.

**Ships with synthetic ladders for both platforms so it runs with no credentials and no network.** Polymarket's public APIs are external and may be blocked in restricted sandboxes; swap in a real pull via `src/polymarket_api.py` and `src/kalshi_api.py` where flagged.

In [ ]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from kalshi_utils import ladder_to_pdf

rng = np.random.default_rng(23)

## 1. Synthetic strike-ladder generator (shared)

Same logistic-survival construction as notebook 01. A `bias` and `extra_noise` knob lets us make the Polymarket series **converge less cleanly** than Kalshi -- the hypothesis Section 5 tests (does CFTC market-maker infrastructure produce a tighter, better-calibrated signal?) -- without asserting it as fact.

In [ ]:
def synthetic_ladder(true_rate, n_days=160, strike_step=0.25, bias=0.0, extra_noise=0.0):
    """Daily mean/median/mode from a strike ladder that noisily converges to true_rate."""
    strikes = np.arange(true_rate - 2.0, true_rate + 2.0 + strike_step, strike_step)
    rows = []
    for days_before in range(n_days, -1, -1):
        noise_scale = (0.5 + extra_noise) * (days_before / n_days) + 0.02
        center = true_rate + bias + rng.normal(0, noise_scale)
        spread = 0.15 + 0.6 * (days_before / n_days)
        yes = 1 / (1 + np.exp((strikes - center) / max(spread, 0.05)))
        yes = np.clip(yes, 0.01, 0.99)
        dist = ladder_to_pdf(list(strikes), list(yes))
        rows.append({"days_before": days_before, "mean": dist.mean,
                     "median": dist.median, "mode": dist.mode})
    return pd.DataFrame(rows)

## 2. Simulate several FOMC meetings on both platforms

Kalshi: tighter (the regulated, market-made book). Polymarket: same true outcomes but a small persistent bias and extra noise, standing in for a thinner, pseudonymous, crypto-native book. **These knobs are assumptions, not measurements** -- the point is the pipeline, which a real pull drops straight into.

In [ ]:
meetings = [4.50, 4.25, 4.25, 4.00, 3.75, 3.75]

def errors_for(platform, bias, extra_noise):
    frames = []
    for true_rate in meetings:
        d = synthetic_ladder(true_rate, bias=bias, extra_noise=extra_noise)
        for col in ["mean", "median", "mode"]:
            frames.append(pd.DataFrame({
                "days_before": d["days_before"],
                "abs_error": (d[col] - true_rate).abs(),
                "series": f"{platform} {col}",
                "platform": platform,
            }))
    return pd.concat(frames, ignore_index=True)

kalshi = errors_for("Kalshi", bias=0.0, extra_noise=0.0)
poly   = errors_for("Polymarket", bias=0.05, extra_noise=0.25)
agg = (pd.concat([kalshi, poly], ignore_index=True)
         .groupby(["platform", "series", "days_before"])["abs_error"].mean().reset_index())
agg.head()

## 3. (Optional) real pull -- swap in here

Guarded no-op offline. On Kalshi, build the daily ladder from candlesticks with `kalshi_utils.candlesticks_to_daily_ladder`; on Polymarket, assemble sibling FOMC markets into a ladder with `polymarket_api.fed_ladder_from_markets`, then run the *same* `candlesticks_to_daily_ladder` (its price history frames already have `date` / `yes_price` columns).

In [ ]:
USE_REAL_DATA = False  # flip to True with Kalshi + Polymarket API access

if USE_REAL_DATA:
    import kalshi_api, polymarket_api
    from kalshi_utils import candlesticks_to_daily_ladder
    # --- Kalshi ---
    # candles_by_strike = { strike: kalshi_api.get_market_candlesticks(...)[['date','yes_price']] ... }
    # kalshi_daily = candlesticks_to_daily_ladder(candles_by_strike)
    # --- Polymarket ---
    # mkts = polymarket_api.list_event_markets('fed-decision-<meeting-slug>')
    # mkts['strike'] = ...  # map each sibling outcome to a rate threshold
    # poly_ladder = polymarket_api.fed_ladder_from_markets(mkts)
    # poly_daily = candlesticks_to_daily_ladder(poly_ladder)
    print("Wire up per comments; then rebuild `agg` from the two *_daily frames.")
else:
    print("Using synthetic ladders (no network). Set USE_REAL_DATA=True with API access.")

## 4. Side-by-side Figure-1-style plot

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
styles = {"Kalshi": "-", "Polymarket": "--"}
colors = {"mean": "tab:blue", "median": "tab:red", "mode": "tab:green"}
for (platform, series_name), g in agg.groupby(["platform", "series"]):
    moment = series_name.split()[-1]
    g = g.sort_values("days_before")
    ax.plot(g["days_before"], g["abs_error"], styles[platform], color=colors[moment],
            linewidth=1.2, label=series_name)
ax.invert_xaxis()
ax.set_xlabel("Days before FOMC")
ax.set_ylabel("Mean absolute forecast error (percent)")
ax.set_title("SYNTHETIC DATA -- Kalshi (solid) vs Polymarket (dashed); replace with real pulls")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

## 5. Confounds -- read before drawing any conclusion

A clean "regulation improves signal quality" claim is **not** identified from this comparison. Kalshi and Polymarket differ in more than regulatory status:

- **User base**: KYC'd, partly-institutional (Kalshi) vs. pseudonymous, crypto-native (Polymarket).
- **Settlement**: USD vs. USDC on-chain.
- **Contract design**: strike *ladder* (Kalshi) vs. *sibling* YES/NO markets you must assemble into a ladder yourself.
- **Liquidity & limits**: CFTC market-maker infrastructure (Susquehanna) and a $7M cap vs. looser limits.
- **History & availability**: different listing dates and data access.

Any observed error gap is therefore **suggestive, not causal**. Report it that way in Section 5 unless the evidence is unusually strong. Eichengreen, Viswanath-Natraj, Wang & Wang (2025) -- cited by the FEDS paper (p.8) -- use Polymarket for Fed-independence questions and are the natural bridge for the divergence-around-controversial-events metric.